<a href="https://colab.research.google.com/github/Srikrishna69420/RandAugment-Implementation/blob/main/RandAugment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

from PIL import Image, ImageOps, ImageEnhance
from torch.utils.data import DataLoader



#RandAugment
In this colab, I will be implementing and verifying the results of RandAugment paper. This paper was revolutionary, as in it simplified the search space massively. Here, the maximum value M can take is 30, therefore it is defined as so. The following functions maps the value of M to the said augmentation magnitude.

In [ ]:
PARAMETER_MAX = 30

def int_parameter(level, maxval):
    return int(level * maxval / PARAMETER_MAX)

def float_parameter(level, maxval):
    return float(level) * maxval / PARAMETER_MAX

def sample_sign():
    return -1 if random.random() < 0.5 else 1

##Augments
In this part, 14 augmentation techniques have been introduced. The only difference is that Cutout was excluded.

In [ ]:
def ShearX(img, level):

    shear = float_parameter(level, 0.3)
    shear *= sample_sign()
    return img.transform(
        img.size,
        Image.AFFINE,
        (1, shear, 0, 0, 1, 0),
        resample=Image.BICUBIC)


def ShearY(img, level):

    shear = float_parameter(level, 0.3)
    shear *= sample_sign()
    return img.transform(
        img.size,
        Image.AFFINE,
        (1, 0, 0, shear, 1, 0),
        resample=Image.BICUBIC)


def TranslateX(img, level):

    max_shift = img.size[0] * 150 / 331
    shift = int_parameter(level, max_shift)
    shift *= sample_sign()
    return img.transform(
        img.size,
        Image.AFFINE,
        (1, 0, shift, 0, 1, 0),
        resample=Image.BICUBIC)


def TranslateY(img, level):

    max_shift = img.size[1] * 150 / 331
    shift = int_parameter(level, max_shift)
    shift *= sample_sign()
    return img.transform(
        img.size,
        Image.AFFINE,
        (1, 0, 0, 0, 1, shift),
        resample=Image.BICUBIC)

def Rotate(img, level):
    degrees = int_parameter(level, 30)
    degrees *= sample_sign()

    return img.rotate(degrees)

def AutoContrast(img, level):
    return ImageOps.autocontrast(img)

def Equalize(img, level):
    return ImageOps.equalize(img)

def Invert(img, level):
    return ImageOps.invert(img)

def Solarize(img, level):
    threshold = 256 - int_parameter(level, 256)

    return ImageOps.solarize(img, threshold)

def Posterize(img, level):
    bits = 4 - int_parameter(level, 4)
    bits = max(bits, 1)

    return ImageOps.posterize(img, bits)

def Contrast(img, level):
    factor = 1.0 + float_parameter(level, 0.9) * sample_sign()

    return ImageEnhance.Contrast(img).enhance(factor)

def Colour(img, level):
    factor = 1.0 + float_parameter(level, 0.9) * sample_sign()

    return ImageEnhance.Color(img).enhance(factor)

def Brightness(img, level):
    factor = 1.0 + float_parameter(level, 0.9) * sample_sign()

    return ImageEnhance.Brightness(img).enhance(factor)

def Sharpness(img, level):
    factor = 1.0 + float_parameter(level, 0.9) * sample_sign()

    return ImageEnhance.Sharpness(img).enhance(factor)

AUGMENT_POOL = [
    ShearX,
    ShearY,
    TranslateX,
    TranslateY,
    Rotate,
    AutoContrast,
    Equalize,
    Invert,
    Solarize,
    Posterize,
    Contrast,
    Colour,
    Brightness,
    Sharpness]

##RandAugment Class
In this part, we implement the RandAugment. Here, I have already assumed that N = 2 and M = 9. In the paper, it is specified that, when N = 2, M = 14, the accuracy was best. However I wanted to experiment, therefore I set N = 2 and M = 9. It is to be noted that the paper suggests marginal difference in accuracy.

In [ ]:
class RandAugment:

    def __init__(self, num_ops=2, magnitude=9):

        self.num_ops = num_ops
        self.magnitude = magnitude

    def __call__(self, img):

        operations = random.choices(
            AUGMENT_POOL,
            k=self.num_ops)

        for op in operations:
            img = op(img, self.magnitude)

        return img

##WideResNet-28-10
In this part, we implement the NN which will be used to find the accuracy. Also label smoothing has been done to prevent overconfidence.

Deoth = 28

Widening factor = 10

In [ ]:
class WideBasic(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride,
        dropout_rate=0.3):

        super().__init__()

        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False)
        self.dropout = nn.Dropout(dropout_rate)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False)
        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False)

    def forward(self, x):

        out = self.conv1(F.relu(self.bn1(x)))
        out = self.dropout(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out += self.shortcut(x)

        return out


class WideResNet(nn.Module):

    def __init__(
        self,
        depth=28,
        widen_factor=10,
        dropout_rate=0.3,
        num_classes=10):

        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        k = widen_factor
        channels = [16, 16*k, 32*k, 64*k]

        self.conv1 = nn.Conv2d(
            3,
            channels[0],
            kernel_size=3,
            padding=1,
            bias=False)
        self.layer1 = self._make_layer(
            channels[0],
            channels[1],
            n,
            stride=1,
            dropout_rate=dropout_rate)
        self.layer2 = self._make_layer(
            channels[1],
            channels[2],
            n,
            stride=2,
            dropout_rate=dropout_rate)
        self.layer3 = self._make_layer(
            channels[2],
            channels[3],
            n,
            stride=2,
            dropout_rate=dropout_rate)
        self.bn = nn.BatchNorm2d(channels[3])
        self.fc = nn.Linear(channels[3], num_classes)
        self._initialize()

    def _make_layer(
        self,
        in_channels,
        out_channels,
        blocks,
        stride,
        dropout_rate):

        layers = []
        for i in range(blocks):

            layers.append(
                WideBasic(
                    in_channels if i == 0 else out_channels,
                    out_channels,
                    stride if i == 0 else 1,
                    dropout_rate))

        return nn.Sequential(*layers)

    def _initialize(self):

        for m in self.modules():

            if isinstance(m, nn.Conv2d):

                nn.init.kaiming_normal_(
                    m.weight,
                    mode='fan_out',
                    nonlinearity='relu')

            elif isinstance(m, nn.BatchNorm2d):

                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):

                nn.init.constant_(m.bias, 0)

    def forward(self, x):

        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn(out))
        out = F.avg_pool2d(out, 8)
        out = out.view(out.size(0), -1)
        out = self.fc(out)

        return out

class LabelSmoothingLoss(nn.Module):

    def __init__(self, smoothing=0.1):

        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):

        num_classes = pred.size(1)
        log_probs = F.log_softmax(pred, dim=1)

        with torch.no_grad():

            true_dist = torch.zeros_like(log_probs)

            true_dist.fill_(self.smoothing / (num_classes - 1))

            true_dist.scatter_(
                1,
                target.unsqueeze(1),
                1.0 - self.smoothing)

        return torch.mean(torch.sum(-true_dist * log_probs, dim=1))

##Configurations

The variables of the paper are defined here. Although epochs were set to 200 in the paper, I have reduced to 100.

In [ ]:
class Config:

    epochs = 100
    batch_size = 128
    lr = 0.1
    momentum = 0.9
    weight_decay = 5e-4
    label_smoothing = 0.1
    num_workers = 4
    randaugment_n = 2
    randaugment_m = 9

    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()

##Data Preprocessing

Here, we preprocess the data, including the normalization of the data with the mean and standard deviation of CIFAR-10 dataset.

In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    RandAugment(
        num_ops=cfg.randaugment_n,
        magnitude=cfg.randaugment_m),
    T.ToTensor(),
    T.Normalize(mean, std)])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std)])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
model = WideResNet(
    depth=28,
    widen_factor=10,
    dropout_rate=0.3).to(cfg.device)

criterion = LabelSmoothingLoss(smoothing=cfg.label_smoothing)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=cfg.lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
    nesterov=True)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.epochs)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_2852/4165279217.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


##Training
In this final part, we train the model and verify the paper.

In [ ]:
best_acc = 0.0

for epoch in range(cfg.epochs):

    model.train()

    for images, labels in train_loader:

        images = images.to(cfg.device)
        labels = labels.to(cfg.device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(cfg.device)
            labels = labels.to(cfg.device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100.0 * correct / total

    if acc > best_acc:
        best_acc = acc

    print(
        f"Epoch [{epoch+1}/{cfg.epochs}] "
        f"Test Accuracy: {acc:.2f}% "
        f"Best: {best_acc:.2f}%")

print(f"\nFinal Best Accuracy: {best_acc:.2f}%")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_2852/2412880035.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/100] Test Accuracy: 50.12% Best: 50.12%
Epoch [2/100] Test Accuracy: 57.08% Best: 57.08%
Epoch [3/100] Test Accuracy: 62.20% Best: 62.20%
Epoch [4/100] Test Accuracy: 68.81% Best: 68.81%
Epoch [5/100] Test Accuracy: 66.88% Best: 68.81%
Epoch [6/100] Test Accuracy: 72.09% Best: 72.09%
Epoch [7/100] Test Accuracy: 75.99% Best: 75.99%
Epoch [8/100] Test Accuracy: 71.42% Best: 75.99%
Epoch [9/100] Test Accuracy: 73.47% Best: 75.99%
Epoch [10/100] Test Accuracy: 68.03% Best: 75.99%
Epoch [11/100] Test Accuracy: 78.70% Best: 78.70%
Epoch [12/100] Test Accuracy: 77.99% Best: 78.70%
Epoch [13/100] Test Accuracy: 77.93% Best: 78.70%
Epoch [14/100] Test Accuracy: 75.90% Best: 78.70%
Epoch [15/100] Test Accuracy: 79.20% Best: 79.20%
Epoch [16/100] Test Accuracy: 81.46% Best: 81.46%
Epoch [17/100] Test Accuracy: 82.48% Best: 82.48%
Epoch [18/100] Test Accuracy: 80.02% Best: 82.48%
Epoch [19/100] Test Accuracy: 83.01% Best: 83.01%
Epoch [20/100] Test Accuracy: 84.21% Best: 84.21%
Epoch [21

#Results and Observations
1) Epoch - 50, Runtime - 147 minutes : Best Accuracy - 89.71%
2) Epoch - 75, Runtime - 219 minutes : Best Accuracy - 93.47%
3) Epoch - 100, Runtime - 293 minutes : Best Accuracy - 96.27%

The paper found that the Best Accuracy for CIFAR-10 trained on Wide-ResNet-28-10 was 97.3%, with the only change being that M = 14 (instead of M = 9) and Epoch = 200.  